# Amazon Product Review Scraper
## Dataset Collection for Sentiment Analysis Assignment

This notebook scrapes product reviews from Amazon India and stores them in CSV format for analysis.

**Requirements:**
- Minimum 100 reviews
- Clean, readable data
- CSV format with review_text column
- No restricted or sensitive content
- Additional columns: rating, reviewer_name, review_date

In [1]:
# Section 1: Import Required Libraries
import subprocess
import sys

packages = ['selenium', 'beautifulsoup4', 'pandas', 'webdriver-manager']
for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Import all required libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime
import json

print("\n✓ All libraries imported successfully!")

✓ selenium already installed
Installing beautifulsoup4...
✓ pandas already installed
✓ webdriver-manager already installed

✓ All libraries imported successfully!


In [26]:
# Section 2: Set Up Selenium WebDriver
print("Setting up Selenium WebDriver...")

options = webdriver.ChromeOptions()
options.add_argument('--start-maximized')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
driver.implicitly_wait(10)

print("✓ WebDriver initialized successfully!")

Setting up Selenium WebDriver...
✓ WebDriver initialized successfully!


In [27]:
# Section 3: Navigate directly to product page
print("Navigating to product page...")

# Direct URL for OnePlus Nord 5 on Amazon India
PRODUCT_URL = "https://www.amazon.in/dp/B0FCML66W9"
SEARCH_PRODUCT = "OnePlus Nord 5"

driver.get(PRODUCT_URL)
time.sleep(3)

# Try to close consent dialog if present
try:
    consent_button = driver.find_element(By.ID, 'sp-cc-accept')
    consent_button.click()
    print("✓ Consent dialog closed")
except:
    print("No consent dialog found")

time.sleep(1)
print(f"✓ Loaded product page: {SEARCH_PRODUCT}")

Navigating to product page...
No consent dialog found
✓ Loaded product page: OnePlus Nord 5


In [28]:
# Section 4: Verify we're on product page and prepare for reviews
print("Verifying product page...")

current_url = driver.current_url
print(f"Current URL: {current_url[:80]}...")

# Wait for page to fully load
time.sleep(2)

print("✓ Ready to access reviews")

Verifying product page...
Current URL: https://www.amazon.in/dp/B0FCML66W9?th=1...
✓ Ready to access reviews


In [29]:
# Section 5: Click on Review Count to Access Reviews Page
print("\nNavigating to reviews page...")

driver.execute_script("window.scrollBy(0, 3000);")
time.sleep(2)

# Click on the review link to go to reviews page
try:
    # Find the review link by ID - this is the clickable anchor
    review_link = driver.find_element(By.ID, 'acrCustomerReviewLink')
    print(f"Found review link")
    driver.execute_script("arguments[0].click();", review_link)
    time.sleep(4)
    print("✓ Navigated to reviews page")
except Exception as e:
    print(f"Could not click review link: {str(e)[:80]}")
    # Try alternative using the span text
    try:
        review_span = driver.find_element(By.ID, 'acrCustomerReviewText')
        parent_link = review_span.find_element(By.XPATH, "..")
        driver.execute_script("arguments[0].click();", parent_link)
        time.sleep(4)
        print("✓ Navigated to reviews page (via parent link)")
    except Exception as e2:
        print(f"Alternative also failed: {str(e2)[:80]}")


Navigating to reviews page...
Found review link
✓ Navigated to reviews page


In [30]:
# Section 6: Extract Reviews from All Pages
print("\nStarting review extraction...\n")

reviews_data = []
review_count = 0
MIN_REVIEWS = 100
page_count = 0
max_pages = 20

try:
    while review_count < MIN_REVIEWS and page_count < max_pages:
        page_count += 1
        print(f"Page {page_count}...", end=" ")
        
        # Parse the current page
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # Find ALL divs that might contain reviews
        # Amazon review structure: look for divs containing review-body spans
        potential_reviews = soup.find_all('span', {'data-hook': 'review-body'})
        print(f"Found {len(potential_reviews)} review bodies", end=" ")
        
        for review_body_elem in potential_reviews:
            if review_count >= MIN_REVIEWS:
                break
            
            try:
                # Get the review text directly
                review_text = review_body_elem.get_text(strip=True)
                
                # Skip if too short
                if not review_text or len(review_text) < 10:
                    continue
                
                # Find parent review container
                review_container = review_body_elem.find_parent('div', {'data-hook': 'review'})
                if not review_container:
                    review_container = review_body_elem.find_parent('div')
                
                # Get rating from the same container
                rating_elem = None
                if review_container:
                    rating_elem = review_container.find('i', {'data-hook': 'review-star-rating'})
                if not rating_elem:
                    rating_elem = review_body_elem.find_parent().find('i', {'data-hook': 'review-star-rating'})
                
                rating_text = rating_elem.get_text(strip=True) if rating_elem else "N/A"
                rating = rating_text.split()[0] if rating_text and rating_text != "N/A" else "N/A"
                
                # Get reviewer name
                reviewer_elem = None
                if review_container:
                    reviewer_elem = review_container.find('a', {'data-hook': 'review-author'})
                if not reviewer_elem:
                    reviewer_elem = review_body_elem.find_parent().find('a', {'data-hook': 'review-author'})
                
                reviewer_name = reviewer_elem.get_text(strip=True) if reviewer_elem else "Anonymous"
                
                # Get review date
                date_elem = None
                if review_container:
                    date_elem = review_container.find('span', {'data-hook': 'review-date'})
                if not date_elem:
                    date_elem = review_body_elem.find_parent().find('span', {'data-hook': 'review-date'})
                
                review_date = date_elem.get_text(strip=True) if date_elem else "N/A"
                
                # Store review
                reviews_data.append({
                    'review_text': review_text,
                    'rating': rating,
                    'reviewer_name': reviewer_name,
                    'review_date': review_date
                })
                review_count += 1
                
            except Exception as inner_e:
                continue
        
        print(f"[Extracted: {review_count}]")
        
        if review_count >= MIN_REVIEWS:
            print(f"\n✓ Reached {MIN_REVIEWS} reviews!")
            break
        
        # Try to navigate to next page
        try:
            next_button = driver.find_element(By.XPATH, "//a[contains(@aria-label, 'Next')]")            
            if next_button.get_attribute('disabled') is None:
                driver.execute_script("arguments[0].click();", next_button)
                time.sleep(2)
            else:
                print("No more pages")
                break
        except:
            print("Could not find next button")
            break
    
    print(f"\n{'='*50}")
    print(f"Total reviews extracted: {review_count}")
    print(f"{'='*50}")

except Exception as e:
    print(f"Error during extraction: {str(e)[:100]}")

finally:
    driver.quit()
    print("\n✓ Browser closed")


Starting review extraction...

Page 1... Found 6 review bodies [Extracted: 6]
Page 2... Found 6 review bodies [Extracted: 12]
Page 3... Found 6 review bodies [Extracted: 18]
Page 4... Found 6 review bodies [Extracted: 24]
Page 5... Found 6 review bodies [Extracted: 30]
Page 6... Found 6 review bodies [Extracted: 36]
Page 7... Found 6 review bodies [Extracted: 42]
Page 8... Found 6 review bodies [Extracted: 48]
Page 9... Found 6 review bodies [Extracted: 54]
Page 10... Found 6 review bodies [Extracted: 60]
Page 11... Found 6 review bodies [Extracted: 66]
Page 12... Found 6 review bodies [Extracted: 72]
Page 13... Found 6 review bodies [Extracted: 78]
Page 14... Found 6 review bodies [Extracted: 84]
Page 15... Found 6 review bodies [Extracted: 90]
Page 16... Found 6 review bodies [Extracted: 96]
Page 17... Found 6 review bodies [Extracted: 100]

✓ Reached 100 reviews!

Total reviews extracted: 100

✓ Browser closed


In [31]:
# Section 7: Create DataFrame and Save CSV
print("Creating DataFrame...")

df = pd.DataFrame(reviews_data)

if len(df) > 0:
    print(f"\n✓ DataFrame created with {len(df)} reviews")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nFirst 5 reviews:")
    print(df.head())
    
    # Save to CSV
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"amazon_reviews_{SEARCH_PRODUCT.replace(' ', '_')}_{timestamp}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"\n✓ CSV saved: {csv_filename}")
else:
    print("No reviews were collected")
    csv_filename = None

Creating DataFrame...

✓ DataFrame created with 100 reviews

Columns: ['review_text', 'rating', 'reviewer_name', 'review_date']

DataFrame shape: (100, 4)

First 5 reviews:
                                         review_text rating reviewer_name  \
0  I’ve been using the OnePlus Nord 5 for a few w...    N/A     Anonymous   
1  Been using it over 2 months now and the experi...    N/A     Anonymous   
2  Decent phone with better build quality and per...    N/A     Anonymous   
3  OnePlus Nord 5Purchased on: 20 January 2026Bes...    N/A     Anonymous   
4  It's value for money. It's one of the best mob...    N/A     Anonymous   

  review_date  
0         N/A  
1         N/A  
2         N/A  
3         N/A  
4         N/A  

✓ CSV saved: amazon_reviews_OnePlus_Nord_5_20260410_101426.csv


In [32]:
# Section 8: Data Cleaning and Validation
if len(df) > 0:
    print("Data Cleaning and Validation\n")
    
    print(f"Original shape: {df.shape}")
    
    # Remove duplicates
    df_clean = df.drop_duplicates(subset=['review_text'], keep='first')
    print(f"After removing duplicates: {df_clean.shape}")
    
    # Remove null values in critical columns
    df_clean = df_clean.dropna(subset=['review_text'])
    
    print(f"Final shape: {df_clean.shape}")
    
    # Display statistics
    print(f"\n{'='*50}")
    print("SUMMARY STATISTICS")
    print(f"{'='*50}")
    print(f"Total reviews: {len(df_clean)}")
    print(f"\nRating distribution:")
    print(df_clean['rating'].value_counts().sort_index())
    print(f"\nReview length stats (characters):")
    print(df_clean['review_text'].str.len().describe())
    print(f"\nNull values per column:")
    print(df_clean.isnull().sum())
    
    # Save cleaned data
    cleaned_filename = csv_filename.replace('.csv', '_cleaned.csv')
    df_clean.to_csv(cleaned_filename, index=False, encoding='utf-8')
    print(f"\n✓ Cleaned CSV saved: {cleaned_filename}")
    
    # Show sample reviews
    print(f"\n{'='*50}")
    print("SAMPLE REVIEWS (First 3)")
    print(f"{'='*50}")
    for idx, row in df_clean.head(3).iterrows():
        print(f"\nReview {idx+1}:")
        print(f"  Rating: {row['rating']} stars")
        print(f"  Reviewer: {row['reviewer_name']}")
        print(f"  Date: {row['review_date']}")
        text_preview = row['review_text'][:150]
        print(f"  Text: {text_preview}...")
        print("-" * 50)
else:
    print("No data to clean!")

Data Cleaning and Validation

Original shape: (100, 4)
After removing duplicates: (6, 4)
Final shape: (6, 4)

SUMMARY STATISTICS
Total reviews: 6

Rating distribution:
rating
N/A    6
Name: count, dtype: int64

Review length stats (characters):
count       6.000000
mean      840.666667
std       653.871445
min       392.000000
25%       396.750000
50%       607.500000
75%       925.500000
max      2079.000000
Name: review_text, dtype: float64

Null values per column:
review_text      0
rating           0
reviewer_name    0
review_date      0
dtype: int64

✓ Cleaned CSV saved: amazon_reviews_OnePlus_Nord_5_20260410_101426_cleaned.csv

SAMPLE REVIEWS (First 3)

Review 1:
  Rating: N/A stars
  Reviewer: Anonymous
  Date: N/A
  Text: I’ve been using the OnePlus Nord 5 for a few weeks now, and honestly, it’s one of the best all‑round mid‑range phones I’ve tried in a long time. The S...
--------------------------------------------------

Review 2:
  Rating: N/A stars
  Reviewer: Anonymous
  